In [30]:
import pandas as pd
import json
import unicodedata
import re


def letter_encoding(tekstas):
    english_letters = unicodedata.normalize('NFD', tekstas)
    english_letters = ''.join(c for c in english_letters if unicodedata.category(c) != 'Mn')
    only_letters = re.sub(r'[^a-zA-Z]', '', english_letters)
    return only_letters

data = pd.read_csv('input_data.csv')

with open('dacia_priority_model_only.json', 'r') as f:
    raw_names = json.load(f)


new_rows = []

group = {
    'Hybrid': ['hybrid', 'hibridas', 'hibrid', 'hybridas'],
    'MCV Stepway': ['stepway', 'stepvay', 'stepvai', 'stepwai'],
    'MCV': ['mcv', 'cmv', 'mvc'],
    'Pick-up': ['pickup', 'picup', 'pikup',  'pikapas']
}

names = []
for model, variant in raw_names.items():
    sorted_variants = dict(sorted(variant.items(), key=lambda item: len(item[0]), reverse=True))
    names.append((model, sorted_variants))


for i, row in data.iterrows():

    model = ""
    variant = ""

    text = str(row['title']).lower()
    text = letter_encoding(text)


    for name, subnames in names:
        if name.lower() in text:
            model = model + name + " "
            text = text.replace(name.lower(), "")

            for models_variant in subnames:
                if models_variant == "<NONE>":
                    continue
                
                keywords = group.get(models_variant, [])

                if any(keyword in text for keyword in keywords):
                    if models_variant not in variant:
                        variant += models_variant + " "
                        break

    if model == "":
        model = "<NONE>"
        variant = "<NONE>"
    elif variant == "":
        variant = "<NONE>"
    

    new_rows.append([row['title'], model, variant])

new_data = pd.DataFrame(new_rows, columns = ['Title', 'Model', 'Variant']) 

new_data.to_csv('output_data.csv', index = False)

print("New CSV created")




New CSV created
